In [ ]:
from pathlib import Path
import subprocess
from Bio import SeqIO



# =====================================================
# مسیرها
# =====================================================

# input ORF 
ORF_DIR = Path(r"E:\......\ORFs")


# database made with makeblastdb
E6_DB = r"/mnt/...../E7_DB"


# output directory
OUTPUT_FASTA = Path(r"E:\.........\E7_all.fasta")


# temporrary files directory
BLAST_DIR = Path(r"E:\..........\E7_blast")

BLAST_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FASTA.parent.mkdir(parents=True, exist_ok=True)


# =====================================================
# settings
# =====================================================

OUTFMT = (
    "6 qseqid sseqid pident length qlen slen "
    "evalue bitscore qstart qend"
)


# =====================================================
# 
# =====================================================

final_records = []


# =====================================================
# 
# =====================================================

faa_files = sorted(ORF_DIR.glob("*_orfs.faa"))

print("Number of genomes:", len(faa_files))


for faa in faa_files:

    genome_name = faa.stem.replace("_orfs", "")

    fna = ORF_DIR / f"{genome_name}_orfs.fna"


    if not fna.exists():
        print("Missing nucleotide file:", genome_name)
        continue


    print("\nProcessing:", genome_name)


    blast_out = BLAST_DIR / f"{genome_name}_E6.tsv"


    # -------------------------------
    # blastp
    # -------------------------------

    cmd = [
        "wsl",
        "blastp",

        "-query",
        "/mnt/e/" + str(faa.relative_to("E:\\")).replace("\\","/"),

        "-db",
        E6_DB,

        "-out",
        "/mnt/e/" + str(blast_out.relative_to("E:\\")).replace("\\","/"),

        "-outfmt",
        OUTFMT,

        "-evalue",
        "1e-5",

        "-max_target_seqs",
        "20"
    ]


    subprocess.run(cmd, check=True)


       # -------------------------------
    # 
    # -------------------------------

    cols = [
        "qseqid",
        "sseqid",
        "pident",
        "length",
        "qlen",
        "slen",
        "evalue",
        "bitscore",
        "qstart",
        "qend"
    ]


    hits = []


    with open(blast_out) as f:

        for line in f:

            values = line.strip().split("\t")

            if len(values) == 10:

                hit = dict(zip(cols, values))

                hit["pident"] = float(hit["pident"])
                hit["length"] = int(hit["length"])
                hit["qlen"] = int(hit["qlen"])
                hit["bitscore"] = float(hit["bitscore"])
                hit["evalue"] = float(hit["evalue"])

                hit["coverage"] = (
                    hit["length"] /
                    hit["qlen"] * 100
                )

                hits.append(hit)


    if len(hits) == 0:
        print("No valid E6 hit:", genome_name)
        continue


    # -------------------------------
    # find best ORF
    # -------------------------------

    best = sorted(
        hits,
        key=lambda x: (
            x["bitscore"],
            x["coverage"],
            x["pident"]
        ),
        reverse=True
    )[0]


    best_orf = best["qseqid"]

    # -------------------------------
    # extract nucleotide sequence of ORF
    # -------------------------------

    found = False


    for record in SeqIO.parse(fna, "fasta"):

        if record.id == best_orf:

            record.id = genome_name + "_E7"
            record.description = (
                f"E7 extracted from {genome_name}"
            )

            final_records.append(record)

            found = True
            break


    if not found:
        print(
            "ORF not found in nucleotide file:",
            best_orf
        )


# =====================================================
# saving
# =====================================================

SeqIO.write(
    final_records,
    OUTPUT_FASTA,
    "fasta"
)


print("\n==============================")
print("Finished")
print("E7 sequences extracted:",
      len(final_records))
print("Saved to:")
print(OUTPUT_FASTA)
print("==============================")

In [ ]:
#notice
# all "E6" and "E7" wich used through the code must be changed based on your dataset.